# Homework 10 - Kubernetes

## Q1. Run it to test that it is working locally

In [1]:
# Q1: docker run -it --rm -p 9696:9696 zoomcamp-model:3.13.10-hw10
# then, in another terminal: python q6_test.py
import requests

url = 'http://localhost:9696/predict'
client = {'job': 'management', 'duration': 400, 'poutcome': 'success'}

response = requests.post(url, json=client).json()
print(response)

answer_q1 = min([0.29, 0.49, 0.69, 0.89],
                key=lambda o: abs(o - response['conversion_probability']))
print('closest option:', answer_q1)
answer_q1


{'conversion_probability': 0.49999999999842815, 'conversion': False}
closest option: 0.49


0.49

## Q2. What is the version of kind?

In [2]:
# Q2: kind --version
import re
import subprocess

out = subprocess.run(['kind', '--version'], capture_output=True, text=True)
print(out.stdout.strip() or out.stderr.strip())

kind_version = re.search(r'(\d+\.\d+\.\d+)', out.stdout).group(1)
kind_version


kind version 0.33.0


'0.33.0'

## Q3. Smallest deployable computing unit in Kubernetes

In [3]:
# Q3: Options: Node, Pod, Deployment, Service
answer_q3 = 'Pod'
answer_q3


'Pod'

## Q4. Type of service already running

In [4]:
# Q4: kubectl get svc
import subprocess

print(subprocess.run(['kubectl', 'get', 'svc'],
                       capture_output=True, text=True).stdout)

service_type = subprocess.run(
    ['kubectl', 'get', 'svc', 'kubernetes', '-o', 'jsonpath={.spec.type}'],
    capture_output=True, text=True,
).stdout.strip()
print('type of the running kubernetes service:', service_type)
service_type


NAME           TYPE           CLUSTER-IP    EXTERNAL-IP   PORT(S)        AGE
kubernetes     ClusterIP      10.96.0.1     <none>        443/TCP        154m
subscription   LoadBalancer   10.96.64.92   <pending>     80:31455/TCP   153m



type of the running kubernetes service:

 ClusterIP


'ClusterIP'

## Q5. Command to register image with kind

In [5]:
# Q5: options: kind create cluster / kind build node-image /
#           kind load docker-image / kubectl apply
cmd = 'kind load docker-image zoomcamp-model:3.13.10-hw10'
print(cmd)

answer_q5 = 'kind load docker-image'
answer_q5


kind load docker-image zoomcamp-model:3.13.10-hw10


'kind load docker-image'

## Q6. Deployment port

In [6]:
# Q6: kubectl apply -f deployment.yaml && kubectl get pods
import pathlib
import subprocess

print(pathlib.Path('deployment.yaml').read_text())

print(subprocess.run(['kubectl', 'apply', '-f', 'deployment.yaml'],
                       capture_output=True, text=True).stdout)
print(subprocess.run(['kubectl', 'get', 'pods'],
                       capture_output=True, text=True).stdout)

port = int(subprocess.run(
    ['kubectl', 'get', 'deploy', 'subscription', '-o',
     'jsonpath={.spec.template.spec.containers[0].ports[0].containerPort}'],
    capture_output=True, text=True,
).stdout.strip())
print('containerPort:', port)
port


apiVersion: apps/v1
kind: Deployment
metadata:
  name: subscription
spec:
  selector:
    matchLabels:
      app: subscription
  replicas: 1
  template:
    metadata:
      labels:
        app: subscription
    spec:
      containers:
      - name: subscription
        image: zoomcamp-model:3.13.10-hw10
        resources:
          requests:
            memory: "64Mi"
            cpu: "100m"
          limits:
            memory: "128Mi"
            cpu: "200m"
        ports:
        - containerPort: 9696


deployment.apps/subscription configured


NAME                            READY   STATUS        RESTARTS   AGE
subscription-7d958b55cf-4k9wp   1/1     Terminating   0          7m40s
subscription-7d958b55cf-r788x   1/1     Terminating   0          7m40s
subscription-7d958b55cf-xcg6s   1/1     Running       0          153m

containerPort: 9696


9696

## Q7. Service selector

In [7]:
# Q7: kubectl apply -f service.yaml
import pathlib
import subprocess

print(pathlib.Path('service.yaml').read_text())

print(subprocess.run(['kubectl', 'apply', '-f', 'service.yaml'],
                       capture_output=True, text=True).stdout)

selector = subprocess.run(
    ['kubectl', 'get', 'svc', 'subscription', '-o',
     'jsonpath={.spec.selector.app}'],
    capture_output=True, text=True,
).stdout.strip()
selector


apiVersion: v1
kind: Service
metadata:
  name: subscription
spec:
  type: LoadBalancer
  selector:
    app: subscription
  ports:
  - port: 80
    targetPort: 9696


service/subscription unchanged



'subscription'

## Q8. Max replicas

In [8]:
# Q8: kubectl autoscale deployment subscription --name subscription-hpa \
#       --cpu-percent=20 --min=1 --max=3
import subprocess
import time

import requests


def sh(args):
    return subprocess.run(args, capture_output=True, text=True).stdout


def current_replicas():
    value = sh(['kubectl', 'get', 'hpa', 'subscription-hpa',
                '-o', 'jsonpath={.status.currentReplicas}']).strip()
    return int(value) if value.isdigit() else 0


print(sh(['kubectl', 'get', 'hpa']))

settle_until = time.time() + 90
while current_replicas() > 1 and time.time() < settle_until:
    time.sleep(5)
print('replicas before load:', current_replicas())

forward = subprocess.Popen(
    ['kubectl', 'port-forward', 'service/subscription', '8081:80'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(3)

url = 'http://localhost:8081/predict'
client = {'job': 'management', 'duration': 400, 'poutcome': 'success'}

max_replicas = current_replicas()
requests_sent = 0
try:
    deadline = time.time() + 120
    next_poll = 0.0
    while time.time() < deadline:
        requests.post(url, json=client)
        requests_sent += 1
        time.sleep(0.02)
        if time.time() >= next_poll:
            max_replicas = max(max_replicas, current_replicas())
            next_poll = time.time() + 5
    max_replicas = max(max_replicas, current_replicas())
finally:
    forward.terminate()

print('requests sent:', requests_sent)
print(sh(['kubectl', 'get', 'hpa']))
print(sh(['kubectl', 'get', 'pods']))
print('max replicas observed:', max_replicas)
max_replicas


NAME               REFERENCE                 TARGETS       MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   cpu: 2%/20%   1         3         3          153m



replicas before load: 1


requests sent: 3830
NAME               REFERENCE                 TARGETS        MINPODS   MAXPODS   REPLICAS   AGE
subscription-hpa   Deployment/subscription   cpu: 39%/20%   1         3         3          155m



NAME                            READY   STATUS    RESTARTS   AGE
subscription-7d958b55cf-7pnnb   1/1     Running   0          2m5s
subscription-7d958b55cf-sm9mk   1/1     Running   0          2m5s
subscription-7d958b55cf-xcg6s   1/1     Running   0          155m

max replicas observed: 3


3